# CallWhisper-8k: Kaggle Whisper-small LoRA Edge Smoke Test

Purpose: start the compact-model adaptation track on Kaggle GPU.

This notebook is intentionally a smoke test first. It proves that the fine-tuning pipeline works, produces a LoRA adapter, and evaluates it on the frozen GramVaani 50-file manifests. Do not report this as a final trained-model result unless the training data is a real training split and excludes the frozen benchmark files.

Serious run target: `GV_Train_100h` or another separate training split.

Frozen held-out eval manifests:

- `datasets/manifests/gramvaani_dev_50.csv`
- `datasets/manifests/gramvaani_dev_50_8khz.csv`
- `datasets/manifests/gramvaani_dev_50_highrate.csv`

Kaggle path rule:

- Uploaded datasets are read-only under `/kaggle/input/<dataset-name>/...`
- Checkpoints and results must be written under `/kaggle/working/...`


## What To Upload To Kaggle

Create one Kaggle Dataset, for example `call-whisper-data`, and attach it to this notebook with **Add Input**.

Minimum smoke-test upload:

```text
GV_Dev_5h/
  Audio/*.mp3
  text
  mp3.scp
  uttids
  utt2labels
```

Recommended serious training upload:

```text
GV_Train_100h/
  Audio/*.mp3
  text
  mp3.scp
  uttids
  utt2labels
GV_Dev_5h/
  Audio/*.mp3
  text
  mp3.scp
  uttids
  utt2labels
```

Optional, only if your local manifests are newer than GitHub:

```text
manifests/
  gramvaani_dev_50.csv
  gramvaani_dev_50_8khz.csv
  gramvaani_dev_50_highrate.csv
```

The repo already contains the frozen manifests, so `manifests/` is optional unless you changed them locally.


In [ ]:
import os
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/anshulLuhsna/CallWhisper-8k.git"
REPO_DIR = Path("/kaggle/working/CallWhisper-8k")
KAGGLE_INPUT_ROOT = Path("/kaggle/input")
WORKING_DIR = Path("/kaggle/working")
CHECKPOINT_DIR = WORKING_DIR / "checkpoints/whisper-small-lora-gramvaani-smoke"
RESULTS_DIR = WORKING_DIR / "results"
LOCAL_DATA_ROOT = WORKING_DIR / "data"

if not REPO_DIR.exists():
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)

os.chdir(REPO_DIR)
os.environ["PYTHONPATH"] = str(REPO_DIR / "src")
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
LOCAL_DATA_ROOT.mkdir(parents=True, exist_ok=True)

print("Repo:", Path.cwd())
print("Kaggle input root:", KAGGLE_INPUT_ROOT)
print("Checkpoint dir:", CHECKPOINT_DIR)
print("Results dir:", RESULTS_DIR)
print("Local data root:", LOCAL_DATA_ROOT)


In [ ]:
!nvidia-smi
!python -m pip install -q -U pip
!python -m pip install -q -e "."
!python -m pip install -q   "transformers==4.46.3"   "peft==0.13.2"   "accelerate==0.34.2"   "datasets==3.1.0"   "evaluate==0.4.3"   "jiwer>=3.0.4"   "librosa>=0.10"   "soundfile>=0.12"   "tensorboard>=2.16"
!which ffmpeg || echo "ffmpeg not found; Kaggle usually has it, but MP3 decode may fail without it."


In [ ]:
from pathlib import Path

print("Available Kaggle inputs:")
if KAGGLE_INPUT_ROOT.exists():
    for p in sorted(KAGGLE_INPUT_ROOT.iterdir()):
        print("-", p)
else:
    print("/kaggle/input does not exist. Are you running this outside Kaggle?")


## Optional: Download GramVaani Directly In Kaggle

If you did not upload `GV_Train_100h`, this cell can download it from OpenSLR into `/kaggle/working/data`. This requires Kaggle notebook internet to be enabled.

The train tarball is about 2.0 GB before extraction. The dev tarball is small and is downloaded too if `GV_Dev_5h` is missing, because the frozen eval manifests need dev audio.


In [ ]:
import tarfile
import urllib.request

DOWNLOAD_GRAMVAANI_IF_MISSING = True
GV_TRAIN_URL = "https://www.openslr.org/resources/118/GV_Train_100h.tar.gz"
GV_DEV_URL = "https://www.openslr.org/resources/118/GV_Dev_5h.tar.gz"


def has_dir_anywhere(dirname: str) -> bool:
    for root in [KAGGLE_INPUT_ROOT, LOCAL_DATA_ROOT]:
        if not root.exists():
            continue
        for child in root.glob("*"):
            if child.name == dirname or (child / dirname).exists():
                return True
    return False


def download_and_extract(url: str, expected_dirname: str) -> None:
    if has_dir_anywhere(expected_dirname):
        print(expected_dirname, "already available")
        return

    if not DOWNLOAD_GRAMVAANI_IF_MISSING:
        print("Skipping download for", expected_dirname)
        return

    archive = LOCAL_DATA_ROOT / Path(url).name
    print("Downloading", url)
    print("To", archive)
    urllib.request.urlretrieve(url, archive)

    print("Extracting", archive)
    with tarfile.open(archive, "r:gz") as tar:
        tar.extractall(LOCAL_DATA_ROOT)
    print("Extracted", expected_dirname, "under", LOCAL_DATA_ROOT)


download_and_extract(GV_DEV_URL, "GV_Dev_5h")
download_and_extract(GV_TRAIN_URL, "GV_Train_100h")


## Link Uploaded Data Into The Repo

The codebase expects data under `datasets/...`. Kaggle datasets are read-only, so this cell creates symlinks from `/kaggle/input/...` into the cloned repo.


In [ ]:
import shutil

DATASETS_DIR = REPO_DIR / "datasets"
MANIFESTS_DIR = DATASETS_DIR / "manifests"


def find_uploaded_dir(name: str) -> Path | None:
    matches = []
    search_roots = []
    if KAGGLE_INPUT_ROOT.exists():
        search_roots.extend(sorted(KAGGLE_INPUT_ROOT.glob("*")))
    if LOCAL_DATA_ROOT.exists():
        search_roots.append(LOCAL_DATA_ROOT)
        search_roots.extend(sorted(LOCAL_DATA_ROOT.glob("*")))
    for root in search_roots:
        candidate = root / name
        if candidate.exists():
            matches.append(candidate)
        if root.name == name and root.exists():
            matches.append(root)
    return matches[0] if matches else None


def link_uploaded_dir(name: str) -> Path | None:
    source = find_uploaded_dir(name)
    if source is None:
        return None
    target = DATASETS_DIR / name
    if target.exists() or target.is_symlink():
        return target
    target.parent.mkdir(parents=True, exist_ok=True)
    os.symlink(source, target)
    return target


def copy_optional_manifests() -> None:
    uploaded = find_uploaded_dir("manifests")
    if uploaded is None:
        return
    MANIFESTS_DIR.mkdir(parents=True, exist_ok=True)
    for src in uploaded.glob("*.csv"):
        dst = MANIFESTS_DIR / src.name
        shutil.copy2(src, dst)
        print("Copied manifest:", src, "->", dst)

for dirname in ["GV_Train_100h", "GV_Dev_5h"]:
    linked = link_uploaded_dir(dirname)
    print(dirname, "->", linked if linked else "not uploaded")

copy_optional_manifests()

GV_TRAIN = DATASETS_DIR / "GV_Train_100h"
GV_DEV = DATASETS_DIR / "GV_Dev_5h"

if GV_TRAIN.exists():
    TRAIN_DATASET_DIR = GV_TRAIN
    SMOKE_TEST_ONLY = False
elif GV_DEV.exists():
    TRAIN_DATASET_DIR = GV_DEV
    SMOKE_TEST_ONLY = True
else:
    raise FileNotFoundError(
        "Upload GV_Train_100h or GV_Dev_5h as a Kaggle Dataset and attach it with Add Input."
    )

print("Train dataset dir:", TRAIN_DATASET_DIR)
print("Smoke test only:", SMOKE_TEST_ONLY)


## Build Train/Eval Rows

Frozen benchmark IDs are excluded from training. This is mandatory, especially if the smoke run uses `GV_Dev_5h`.


In [ ]:
import csv
import random
from pathlib import Path

SEED = 0
random.seed(SEED)

# Keep this small for first Kaggle run. Increase after one clean smoke pass.
MAX_TRAIN_SAMPLES = 200 if SMOKE_TEST_ONLY else 2000
MAX_EVAL_SAMPLES = 25 if SMOKE_TEST_ONLY else 100
MAX_STEPS = 40 if SMOKE_TEST_ONLY else 500

FROZEN_MANIFESTS = [
    MANIFESTS_DIR / "gramvaani_dev_50.csv",
    MANIFESTS_DIR / "gramvaani_dev_50_8khz.csv",
    MANIFESTS_DIR / "gramvaani_dev_50_highrate.csv",
]


def read_text(path: Path) -> dict[str, str]:
    rows = {}
    with path.open(encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            utt, _, text = line.partition(" ")
            if utt and text:
                rows[utt] = text.strip()
    return rows


def read_scp(path: Path, dataset_dir: Path) -> dict[str, Path]:
    rows = {}
    with path.open(encoding="utf-8") as f:
        for line in f:
            parts = line.strip().split(maxsplit=1)
            if len(parts) != 2:
                continue
            audio_path = Path(parts[1])
            if not audio_path.is_absolute():
                audio_path = dataset_dir / audio_path
            rows[parts[0]] = audio_path
    return rows


def frozen_ids_from_manifest(path: Path) -> set[str]:
    ids = set()
    if not path.exists():
        print("Missing frozen manifest:", path)
        return ids
    with path.open(encoding="utf-8") as f:
        reader = csv.DictReader(f)
        for row in reader:
            audio_path = row.get("audio_path", "")
            if audio_path:
                ids.add(Path(audio_path).stem)
    return ids


def build_rows(dataset_dir: Path) -> list[dict[str, str]]:
    text_path = dataset_dir / "text"
    scp_path = dataset_dir / "mp3.scp"
    if not text_path.exists() or not scp_path.exists():
        raise FileNotFoundError(f"Expected Kaldi-style text and mp3.scp in {dataset_dir}")

    texts = read_text(text_path)
    audio_paths = read_scp(scp_path, dataset_dir)
    frozen_ids = set().union(*(frozen_ids_from_manifest(p) for p in FROZEN_MANIFESTS))

    rows = []
    skipped_frozen = 0
    skipped_missing = 0
    for utt, text in texts.items():
        if utt in frozen_ids:
            skipped_frozen += 1
            continue
        audio_path = audio_paths.get(utt)
        if audio_path is None or not audio_path.exists():
            skipped_missing += 1
            continue
        rows.append({"utt_id": utt, "audio": str(audio_path), "text": text})

    random.shuffle(rows)
    print("Total usable rows:", len(rows))
    print("Skipped frozen benchmark rows:", skipped_frozen)
    print("Skipped missing audio rows:", skipped_missing)
    return rows

all_rows = build_rows(TRAIN_DATASET_DIR)
if len(all_rows) < 10:
    raise RuntimeError("Too few rows found. Check your uploaded GV folder structure.")

train_rows = all_rows[:MAX_TRAIN_SAMPLES]
eval_rows = all_rows[MAX_TRAIN_SAMPLES:MAX_TRAIN_SAMPLES + MAX_EVAL_SAMPLES]
if len(eval_rows) == 0:
    eval_rows = train_rows[-min(len(train_rows), MAX_EVAL_SAMPLES):]
    train_rows = train_rows[:-len(eval_rows)] or train_rows

print("Train rows:", len(train_rows))
print("Eval rows:", len(eval_rows))
print("Max steps:", MAX_STEPS)
print("First row:", train_rows[0])


## Prepare Whisper-small + LoRA

This adapts only a small number of attention projection weights. It is the edge-compute track, not a claim that the model is better than large Hindi-tuned models.


In [ ]:
import numpy as np
import torch
from datasets import Audio, Dataset
from peft import LoraConfig, get_peft_model
from transformers import WhisperForConditionalGeneration, WhisperProcessor

BASE_MODEL = "openai/whisper-small"
processor = WhisperProcessor.from_pretrained(BASE_MODEL, language="Hindi", task="transcribe")
model = WhisperForConditionalGeneration.from_pretrained(BASE_MODEL)
model.config.forced_decoder_ids = None
model.config.suppress_tokens = []
model.config.use_cache = False

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

train_ds = Dataset.from_list(train_rows).cast_column("audio", Audio(sampling_rate=16000))
eval_ds = Dataset.from_list(eval_rows).cast_column("audio", Audio(sampling_rate=16000))
print(train_ds)
print(eval_ds)


In [ ]:
def decode_audio_for_datasets(audio):
    # datasets 3.x can return either a dict or a torchcodec AudioDecoder-like object.
    if isinstance(audio, dict):
        return audio["array"], audio["sampling_rate"]
    if hasattr(audio, "get_all_samples"):
        samples = audio.get_all_samples()
        array = samples.data
        if hasattr(array, "detach"):
            array = array.detach().cpu().numpy()
        array = np.asarray(array).squeeze()
        return array, int(samples.sample_rate)
    raise TypeError(f"Unsupported audio object: {type(audio)}")


def prepare_example(batch):
    array, sampling_rate = decode_audio_for_datasets(batch["audio"])
    batch["input_features"] = processor.feature_extractor(
        array,
        sampling_rate=sampling_rate,
    ).input_features[0]
    batch["labels"] = processor.tokenizer(batch["text"]).input_ids
    return batch

train_ds = train_ds.map(prepare_example, remove_columns=train_ds.column_names)
eval_ds = eval_ds.map(prepare_example, remove_columns=eval_ds.column_names)
print(train_ds)
print(eval_ds)


## Train Smoke Adapter

Run this once as-is. If it completes cleanly, increase `MAX_TRAIN_SAMPLES` and `MAX_STEPS` in the earlier cell.


In [ ]:
import inspect
from dataclasses import dataclass
from typing import Any
from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments

@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any

    def __call__(self, features):
        input_features = [{"input_features": f["input_features"]} for f in features]
        batch = self.processor.feature_extractor.pad(input_features, return_tensors="pt")

        label_features = [{"input_ids": f["labels"]} for f in features]
        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors="pt")
        labels = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)

        if (labels[:, 0] == self.processor.tokenizer.bos_token_id).all().cpu().item():
            labels = labels[:, 1:]

        batch["labels"] = labels
        return batch

args_kwargs = dict(
    output_dir=str(CHECKPOINT_DIR),
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=2,
    learning_rate=1e-4,
    warmup_steps=min(20, max(1, MAX_STEPS // 10)),
    max_steps=MAX_STEPS,
    gradient_checkpointing=True,
    fp16=torch.cuda.is_available(),
    logging_steps=10,
    eval_steps=max(10, MAX_STEPS // 2),
    save_steps=max(10, MAX_STEPS // 2),
    save_total_limit=2,
    predict_with_generate=False,
    report_to=["tensorboard"],
    remove_unused_columns=False,
    label_names=["labels"],
)

sig = inspect.signature(Seq2SeqTrainingArguments)
if "eval_strategy" in sig.parameters:
    args_kwargs["eval_strategy"] = "steps"
else:
    args_kwargs["evaluation_strategy"] = "steps"

training_args = Seq2SeqTrainingArguments(**args_kwargs)

trainer = Seq2SeqTrainer(
    args=training_args,
    model=model,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    data_collator=DataCollatorSpeechSeq2SeqWithPadding(processor=processor),
    tokenizer=processor.feature_extractor,
)

trainer.train()
trainer.save_model(str(CHECKPOINT_DIR / "final_adapter"))
processor.save_pretrained(str(CHECKPOINT_DIR / "processor"))

print("Saved adapter:", CHECKPOINT_DIR / "final_adapter")
print("Saved processor:", CHECKPOINT_DIR / "processor")


## Evaluate Adapter On Frozen GramVaani Manifests

This writes JSON and Markdown summaries into `/kaggle/working/results`. These are Kaggle notebook outputs, so they are saved after the run finishes.


In [ ]:
import json
import librosa
import pandas as pd
from peft import PeftModel
from tqdm.auto import tqdm
from callwhisper.eval.wer import word_error_rate
from callwhisper.eval.cer import character_error_rate

RUN_FROZEN_EVAL = True
EVAL_LIMIT = None  # set to 10 for a quick debug pass
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


def resolve_manifest_audio(path_value: str) -> Path:
    p = Path(path_value)
    if p.is_absolute():
        return p
    return REPO_DIR / p


def transcribe_one(model, audio_path: Path) -> str:
    array, sr = librosa.load(str(audio_path), sr=16000, mono=True)
    inputs = processor.feature_extractor(array, sampling_rate=16000, return_tensors="pt")
    input_features = inputs.input_features.to(DEVICE)
    with torch.no_grad():
        pred_ids = model.generate(
            input_features,
            language="hi",
            task="transcribe",
            max_new_tokens=225,
        )
    return processor.batch_decode(pred_ids, skip_special_tokens=True)[0].strip()


if RUN_FROZEN_EVAL:
    base_model = WhisperForConditionalGeneration.from_pretrained(BASE_MODEL).to(DEVICE)
    adapted_model = PeftModel.from_pretrained(base_model, CHECKPOINT_DIR / "final_adapter")
    adapted_model = adapted_model.merge_and_unload().to(DEVICE)
    adapted_model.eval()

    summaries = []
    for manifest_path in FROZEN_MANIFESTS:
        if not manifest_path.exists():
            print("Skipping missing manifest:", manifest_path)
            continue
        rows = list(csv.DictReader(manifest_path.open(encoding="utf-8")))
        if EVAL_LIMIT is not None:
            rows = rows[:EVAL_LIMIT]

        predictions = []
        wers = []
        cers = []
        for row in tqdm(rows, desc=manifest_path.stem):
            audio_path = resolve_manifest_audio(row["audio_path"])
            hypothesis = transcribe_one(adapted_model, audio_path)
            reference = row["reference_text"]
            wer_value = word_error_rate(reference, hypothesis)
            cer_value = character_error_rate(reference, hypothesis)
            wers.append(wer_value)
            cers.append(cer_value)
            predictions.append({
                **row,
                "audio_path": str(audio_path),
                "hypothesis": hypothesis,
                "wer": wer_value,
                "cer": cer_value,
            })

        result = {
            "model": "openai/whisper-small+lora",
            "base_model": BASE_MODEL,
            "adapter": str(CHECKPOINT_DIR / "final_adapter"),
            "slice": manifest_path.stem,
            "condition": rows[0].get("condition", "unknown") if rows else "unknown",
            "files": len(rows),
            "wer": float(sum(wers) / len(wers)) if wers else None,
            "cer": float(sum(cers) / len(cers)) if cers else None,
            "smoke_test_only": SMOKE_TEST_ONLY,
            "train_dataset_dir": str(TRAIN_DATASET_DIR),
            "max_train_samples": MAX_TRAIN_SAMPLES,
            "max_steps": MAX_STEPS,
            "predictions": predictions,
        }

        out_json = RESULTS_DIR / f"kaggle_whisper_small_lora_{manifest_path.stem}_seed{SEED}.json"
        out_json.write_text(json.dumps(result, ensure_ascii=False, indent=2), encoding="utf-8")
        summaries.append({
            "file": str(out_json),
            "model": result["model"],
            "slice": result["slice"],
            "files": result["files"],
            "wer": result["wer"],
            "cer": result["cer"],
            "smoke_test_only": result["smoke_test_only"],
        })
        print("Wrote", out_json)

    summary_df = pd.DataFrame(summaries)
    summary_path = RESULTS_DIR / "kaggle_whisper_small_lora_summary.md"
    summary_path.write_text(summary_df.to_markdown(index=False), encoding="utf-8")
    print(summary_df.to_markdown(index=False))
    print("Summary:", summary_path)


## Save/Download Outputs

Kaggle automatically keeps files under `/kaggle/working` as notebook outputs.

Important output paths:

```text
/kaggle/working/checkpoints/whisper-small-lora-gramvaani-smoke/final_adapter/
/kaggle/working/checkpoints/whisper-small-lora-gramvaani-smoke/processor/
/kaggle/working/results/kaggle_whisper_small_lora_summary.md
/kaggle/working/results/kaggle_whisper_small_lora_*.json
```

After the run, open the Kaggle notebook **Output** panel and download the checkpoint/results, or create a Kaggle Dataset from the notebook output if you want to reuse the adapter in another notebook.
